PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

In [ ]:
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 8)

# Working with Protein Complexes

`ProteinComplex` is the main class in PePy. It extends BioPandas' `PandasPdb` with chain assignment, interface calculation, and confidence metrics.

This tutorial covers loading structures, identifying chains, calculating interfaces, and accessing the resulting data as DataFrames.

In [ ]:
from pepy import ProteinComplex

## Loading Structures

PePy accepts `.pdb` and `.cif` files. The easiest way is the `from_file` class method:

In [ ]:
cx = ProteinComplex.from_file('../../pepy/tests/data/1YCR.pdb')
cx.df['ATOM'].head()

You can also load CIF files — AF3 CIF files with missing columns are fixed automatically:

In [ ]:
cx_cif = ProteinComplex.from_file('../../pepy/tests/data/fold_1ycr_af3_model_0.cif')
cx_cif.df['ATOM'].head()

Or fetch directly from the PDB:

In [ ]:
cx_fetch = ProteinComplex()
cx_fetch.fetch_pdb('1ycr')
print('Chains:', cx_fetch.get_available_chains())

## Inspecting the Structure

Before assigning chains, it's useful to see what's in the file:

In [ ]:
cx = ProteinComplex.from_file('../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb')

print('Available chains:', cx.get_available_chains())
print('Chain lengths:', cx.get_chain_lengths())

## Identifying Chains

You need to tell PePy which chain(s) are the **binder** and which are the **receptor** before calculating the interface.

### Auto-detection

By default, the shortest chain becomes the binder:

In [ ]:
cx.identify_chains()

print(f'Binder:   {cx.binder_chains}')
print(f'Receptor: {cx.receptor_chains}')

### Explicit assignment

You can specify the binder as a string (single chain) or list (multi-chain):

In [ ]:
# Single-chain binder
cx.identify_chains(binder_chains='B', receptor_chains=['A'])

# Multi-chain binder (if applicable)
# cx.identify_chains(binder_chains=['B', 'C'], receptor_chains=['A'])

### Chain info

Get detailed information about all chains — lengths, roles, sequences:

In [ ]:
info = cx.get_chain_info()
for chain_id, chain_data in info.items():
    print(f"Chain {chain_id}: {chain_data['length']} residues, "
          f"role={chain_data['type']}, seq={chain_data['sequence'][:30]}...")

## Calculating the Interface

The interface calculation uses a two-stage algorithm:

1. **CB prefiltering** — finds residues whose CB atoms (CA for glycine) are within `cb_cutoff` Å
2. **All-atom refinement** — among those, finds residues with any atom within `all_atom_cutoff` Å

Either stage can be disabled by setting its cutoff to `-1`.

In [ ]:
binder_res, receptor_res = cx.calculate_interface(
    cb_cutoff=8.0,          # Å, CB prefilter
    all_atom_cutoff=4.0,    # Å, all-atom refinement
)

print(f'Binder interface residues:   {binder_res}')
print(f'Receptor interface residues: {receptor_res}')

### Optional: confidence filtering

You can filter out low-confidence binder residues based on their pLDDT (stored in the B-factor column):

In [ ]:
binder_res_filtered, receptor_res_filtered = cx.calculate_interface(
    drop_low_confidence=True,
    confidence_threshold=70.0,
)

print(f'Before filtering: {len(binder_res)} binder residues')
print(f'After filtering:  {len(binder_res_filtered)} binder residues')

### Interface summary

In [ ]:
cx.calculate_interface()  # recalculate without filtering
cx.get_interface_summary()

## Accessing Interface Atoms

After calculating the interface, you can get the actual atoms as pandas DataFrames — useful for downstream analysis, visualization, or export.

In [ ]:
# All interface atoms
binder_atoms, receptor_atoms = cx.get_interface_atoms('all')
print(f'Binder interface atoms:   {len(binder_atoms)}')
print(f'Receptor interface atoms: {len(receptor_atoms)}')

binder_atoms.head()

In [ ]:
# Filter by atom type: 'ca', 'backbone', 'sidechain'
binder_ca, receptor_ca = cx.get_interface_atoms('ca')
binder_bb, receptor_bb = cx.get_interface_atoms('backbone')
binder_sc, receptor_sc = cx.get_interface_atoms('sidechain')

print(f'CA only:    {len(binder_ca)} + {len(receptor_ca)}')
print(f'Backbone:   {len(binder_bb)} + {len(receptor_bb)}')
print(f'Sidechain:  {len(binder_sc)} + {len(receptor_sc)}')

These are standard pandas DataFrames — you can select columns, merge, export to CSV, etc.:

In [ ]:
binder_ca[['chain_id', 'residue_number', 'residue_name', 'x_coord', 'y_coord', 'z_coord', 'b_factor']]

## Clearing and Recalculating

Re-assigning chains automatically clears previous interface results:

In [ ]:
print('Interface calculated:', cx.interface_calculated)

# Swap binder and receptor
cx.identify_chains(binder_chains='A', receptor_chains=['B'])
print('After re-identifying chains:', cx.interface_calculated)

# Recalculate with new assignment
binder_res, receptor_res = cx.calculate_interface()
print(f'Now binder (chain A) has {len(binder_res)} interface residues')

You can also clear results manually:

In [ ]:
cx.clear_interface_results()
print('Interface calculated:', cx.interface_calculated)

cx.clear_cache()  # clears everything including chain length cache